<a href="https://colab.research.google.com/github/rakasiwisurya/pgaibllm-lessons/blob/main/Latihan_Advanced_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Persiapan Dependencies dan Ekosistem

In [1]:
!pip install -q transformers==4.57.3
!pip install -q accelerate==1.12.0
!pip install -q bitsandbytes==0.49.1
!pip install -q langchain==1.2.3
!pip install -q langchain-community==0.4.1
!pip install -q langchain-core==1.2.6
!pip install -q langchain-text-splitters
!pip install -q pymupdf
!pip install -q langchain-huggingface
!pip install -q chromadb==1.4.1
!pip install -q huggingface-hub==0.36.0
!pip install -q rank-bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 764.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.5 MB/s eta 0:00:00

In [2]:
import os
import torch
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever, EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryByteStore
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [3]:
!wget -O buku_panduan_gen_ai.pdf "https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb"

--2026-07-20 10:44:32--  https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.google.com (drive.google.com)... 142.251.107.113, 142.251.107.102, 142.251.107.100, ...
Connecting to drive.google.com (drive.google.com)|142.251.107.113|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb [following]
--2026-07-20 10:44:32--  https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 172.217.204.132, 2607:f8b0:400c:c3d::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|172.217.204.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 655208 (640K) [application/octet-stream]
Saving to: ‘buku_panduan_gen_ai.pdf’

buku_panduan_gen_ai 100%[===================>] 639.85K  --.-KB/s    in 0.01s   

2026-07-20 10:44:

In [4]:
file_path = "/content/buku_panduan_gen_ai.pdf"
loader = PyMuPDFLoader(file_path)
documents = loader.load()

# Retriever dengan Parent-Child Document Retriever

In [5]:
# Parent Chunk
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
# Child Chunk
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan device: {device}")

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    # model_kwargs={'device': 'cuda'}
    model_kwargs={'device': device}
)

Menggunakan device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [7]:
vectorstore = Chroma(
    collection_name="split_parents",
    embedding_function=embedding_model
)

/tmp/ipykernel_417/3269059110.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [8]:
# Store untuk Parent
docstore = InMemoryByteStore()

In [9]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_type="similarity",
    search_kwargs={"k": 10}
)

# Masukkan Data
retriever.add_documents(documents)

In [10]:
query = "Framework apa yang dapat digunakan untuk memastikan penggunaan GenAI yang efektif?"
relevant_docs = retriever.invoke(query)

for i, doc in enumerate(relevant_docs, start=1):
    print(f"--- Dokumen {i} ---")
    print(doc.page_content)
    print()

--- Dokumen 1 ---
17
3.3. T.U.C.E. Framework (Think, Use, Check, En-
hance)
Untuk memastikan penggunaan GenAI yang efektif dan bertanggung jawab, maha-
siswa dapat mengingat Framework berikut ini:
1.	 Think (Pikirkan Sebelum Menggunakan GenAI)
a.	 Tentukan tujuan penggunaan GenAI: Apakah GenAI benar-benar diperlukan?
b.	 Pastikan GenAI digunakan sebagai alat bantu, bukan pengganti pemikiran kritis.
c.	 Periksa aturan mata kuliah atau dosen terkait penggunaan GenAI dalam tugas.
2.	 Use (Gunakan GenAI dengan Bijak)
a.	 Pilih alat GenAI yang sesuai dengan kebutuhan (ChatGPT untuk teks, DALL-E 
untuk gambar, dll.).
b.	 Masukkan instruksi yang jelas dan spesifik agar hasilnya relevan.
c.	 Hindari memasukkan data pribadi atau informasi sensitif ke dalam GenAI.
3.	 Check (Periksa Hasil dan Verifikasi Informasi)
a.	 Cek keakuratan dan kredibilitas hasil GenAI dengan sumber akademik yang valid.
b.	 Pastikan tidak ada plagiarisme dan berikan atribusi jika diperlukan.
c.	 Evaluasi apakah GenAI me

# Tambahkan dengan Hybrid Retriever

In [11]:
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 10

In [12]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever],
    weights=[0.5, 0.5]
)

In [13]:
query = "Apa sih itu GenAI detector"
hybrid_relevant_docs = hybrid_retriever.invoke(query)

for i, doc in enumerate(hybrid_relevant_docs, start=1):
    print(f"--- Dokumen {i} ---")
    print(doc.page_content)
    print()

--- Dokumen 1 ---
31
BAB V: Tanya Jawab Seputar 
Penggunaan Generative AI Dalam Akademik
Bagian ini berisi pertanyaan umum yang sering diajukan mahasiswa terkait penggu-
naan GenAI dalam lingkungan akademik, serta batasan dan aturan spesifik yang ber-
laku.
1.	 Kapan mahasiswa boleh menggunakan GenAI dalam tugas akademik? 
Mahasiswa diperbolehkan menggunakan GenAI dalam tugas akademik jika:
1.	 GenAI digunakan sebagai alat bantu dalam brainstorming, merangkum, atau 
menyusun ide.
2.	 GenAI membantu mengoreksi tata bahasa dan meningkatkan kualitas tulisan.
3.	 Mahasiswa tetap memahami dan menyesuaikan hasil GenAI dengan analisis dan 
pemikiran mereka sendiri.
4.	 Sumber GenAI dicantumkan dengan format kutipan yang sesuai.
2.	 Kapan GenAI tidak boleh digunakan dalam tugas akademik?
Mahasiswa tidak diperbolehkan menggunakan GenAI dalam tugas akademik jika:
1.	 Dosen secara eksplisit melarang penggunaan GenAI dalam tugas tertentu.
2.	 Mahasiswa mengandalkan GenAI sepenuhnya tanpa memahami 

# Re-rangking dengan Cross Encoder

In [14]:
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

compressor = CrossEncoderReranker(model=model, top_n=5)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [15]:
final_docs = compression_retriever.invoke(query)

for i, doc in enumerate(final_docs, start=1):
    print(f"--- Dokumen {i} ---")
    print(doc.page_content)
    print()

--- Dokumen 1 ---
25
BAB IV: Ruang Lingkup Integritas Akademik 
Dalam Penggunaan Generative AI
Penggunaan Generative AI (GenAI) dalam lingkungan akademik harus tetap menjun-
jung tinggi integritas akademik, yaitu prinsip kejujuran, tanggung jawab, dan etika da-
lam belajar dan berkarya. Untuk memastikan hal ini, diperlukan strategi yang mencak-
up pencegahan, pembinaan, dan penanggulangan terhadap penyalahgunaan AI dalam 
tugas akademik.
4.1. Pencegahan: Mencegah Penyalahgunaan Gen-
erative AI Sejak Awal
4.1.1. Memahami Kemampuan Dosen dalam Mengetahui Penggunaan Genera-
tive AI pada Mahasiswa
Mahasiswa perlu menyadari bahwa dosen memiliki berbagai cara untuk mendeteksi 
penggunaan GenAI dalam tugas akademik. Beberapa metode yang digunakan meliputi:
1.	 Penggunaan GenAI Detector: Dosen dapat menggunakan alat pendeteksi GenAI 
seperti Turnitin AI Detector, GPTZero, dan Originality.ai untuk mengidentifikasi 
teks yang dihasilkan oleh GenAI.
2.	 Analisis Gaya Tulisan: Dosen dapat membandi

# Generation

In [16]:
# Konfigurasi Kuantisasi 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "unsloth/Llama-3.2-3B-Instruct"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [17]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=1000,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Device set to use cpu


In [18]:
template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Anda adalah asisten AI yang bertugas membantu pengguna.
Gunakan hanya informasi yang tersedia pada konteks berikut untuk menjawab pertanyaan.
Jika jawaban tidak ditemukan dalam konteks tersebut, sampaikan dengan jujur bahwa Anda tidak mengetahui jawabannya dan jangan membuat asumsi atau jawaban tambahan.
Berikan jawaban secara singkat dan jelas.

Context:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [19]:
rag_chain = (
    {"context": compression_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
def ask_question(query):
    print(f"Pertanyaan: {query}")

    # Jalankan Chain
    response = rag_chain.invoke(query)

    print("Jawaban:")
    print(response)

    # Tampilkan sumber referensi
    docs = retriever.invoke(query)
    print("\nSumber Referensi:")
    for i, doc in enumerate(docs):
        print(f"{i+1}. Halaman {doc.metadata.get('page', '?')}")

In [21]:
query = "Apa itu Generative AI?"
ask_question(query)

Pertanyaan: Apa itu Generative AI?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Jawaban:
Generative AI, atau Singkatnya GenAI, adalah sebuah cabang dari Artificial Intelligence (AI) yang dirancang untuk menghasilkan konten baru, seperti teks, gambar, suara, dan video, berdasarkan data yang telah dipelajarinya.

Sumber Referensi:
1. Halaman 9
2. Halaman 8
3. Halaman 38
4. Halaman 11
5. Halaman 12
